# Nebula Qwen3 4B Defensive-Security QLoRA
Run all cells with a GPU runtime. Select `nebula-qwen3-4b-cyber-colab-bundle.zip` when prompted. Checkpoints persist in Google Drive and rerunning resumes safely.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import shutil, zipfile
drive.mount('/content/drive')
uploaded = files.upload()
bundle_name = next((name for name in uploaded if name.endswith('.zip')), None)
if not bundle_name:
    raise RuntimeError('Select nebula-qwen3-4b-cyber-colab-bundle.zip')
root = Path('/content/nebula-qwen3')
if root.exists():
    shutil.rmtree(root)
root.mkdir(parents=True)
with zipfile.ZipFile(bundle_name) as archive:
    archive.extractall(root)
%cd /content/nebula-qwen3
print('Bundle extracted:', bundle_name)

In [ ]:
# Pin a stable CUDA 12.8 stack; Colab preview images may otherwise install CUDA 13 nightly builds.
!pip -q uninstall -y torch torchvision torchaudio bitsandbytes
!pip -q install torch==2.7.1 --index-url https://download.pytorch.org/whl/cu128
!pip -q install -U -r training/requirements-qlora.txt
!pip -q uninstall -y torchvision torchao
import torch
assert torch.cuda.is_available(), 'Switch Colab to a GPU runtime before training.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'training/scripts/validate_dataset.py', '--train', 'training/qwen3/data/train.jsonl', '--validation', 'training/qwen3/data/validation.jsonl', '--minimum-examples', '4000', '--allow-tool', 'run_command', '--heldout', 'training/evals/qwen_coder_behavior.jsonl', '--heldout', 'training/evals/qwen_coder_stress.jsonl', '--heldout', 'training/evals/qwen3_cyber_behavior.jsonl', '--report', 'training/qwen3/data/validation-report.json'], check=True)

In [ ]:
from pathlib import Path
import subprocess, sys
output = Path('/content/drive/MyDrive/NebulaTraining/nebula-qwen3-4b-cyber-v1/adapter')
output.mkdir(parents=True, exist_ok=True)
print('Checkpoints:', output)
subprocess.run([sys.executable, 'training/scripts/train_qlora.py', '--config', 'training/configs/qwen3-4b-nebula-cyber-qlora.json', '--output-dir', str(output), '--resume'], check=True)

In [ ]:
from pathlib import Path
adapter = Path('/content/drive/MyDrive/NebulaTraining/nebula-qwen3-4b-cyber-v1/adapter')
required = ['adapter_config.json', 'adapter_model.safetensors']
missing = [name for name in required if not (adapter / name).exists()]
if missing:
    raise RuntimeError(f'Training did not produce final adapter files: {missing}')
print('Training complete. Adapter:', adapter)